# Hyperopt (Hyperparameter Optimization)

In [4]:
# Install required packages
# pip install hyperopt xgboost scikit-learn

import numpy as np
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb

# Load the Iris dataset
iris = load_iris()
X = iris.data  # Features: sepal length, sepal width, petal length, petal width
y = iris.target  # Target: species (0, 1, 2)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [10]:
# Define the hyperparameter search space
space = {
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'n_estimators': hp.quniform('n_estimators', 50, 200, 10),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'reg_lambda': hp.uniform('reg_lambda', 0.1, 2.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1.0),
}

In [12]:
def objective(params):
    # Convert parameters to appropriate types
    params = {
        'max_depth': int(params['max_depth']),
        'learning_rate': params['learning_rate'],
        'n_estimators': int(params['n_estimators']),
        'subsample': params['subsample'],
        'colsample_bytree': params['colsample_bytree'],
        'reg_lambda': params['reg_lambda'],
        'reg_alpha': params['reg_alpha'],
        'random_state': 42,
        'n_jobs': -1
    }

    # Create and train model
    model = xgb.XGBClassifier(**params)

    # Use cross-validation for robust evaluation
    score = cross_val_score(model, X_train, y_train,
                          scoring='accuracy', cv=3, n_jobs=-1)
    score = score.mean()




    # Hyperopt minimizes, so we return negative accuracy
    return {'loss': -score, 'status': STATUS_OK}

In [13]:
# Initialize trials object to track progress
trials = Trials()

# Run optimization
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,  # Number of iterations
    trials=trials
)

print("\nBest parameters found:")
print(best)

100%|██████████| 50/50 [00:10<00:00,  4.74trial/s, best loss: -0.975]

Best parameters found:
{'colsample_bytree': np.float64(0.6236835736701225), 'learning_rate': np.float64(0.08863791136259769), 'max_depth': np.float64(4.0), 'n_estimators': np.float64(190.0), 'reg_alpha': np.float64(0.9946955148850362), 'reg_lambda': np.float64(1.3448672246712803), 'subsample': np.float64(0.6602731704156627)}


In [14]:
trials

In [15]:
best

{'colsample_bytree': np.float64(0.6236835736701225),
 'learning_rate': np.float64(0.08863791136259769),
 'max_depth': np.float64(4.0),
 'n_estimators': np.float64(190.0),
 'reg_alpha': np.float64(0.9946955148850362),
 'reg_lambda': np.float64(1.3448672246712803),
 'subsample': np.float64(0.6602731704156627)}

In [16]:
# Convert best parameters to proper types
best_params = {
    'max_depth': int(best['max_depth']),
    'learning_rate': best['learning_rate'],
    'n_estimators': int(best['n_estimators']),
    'subsample': best['subsample'],
    'colsample_bytree': best['colsample_bytree'],
    'reg_lambda': best['reg_lambda'],
    'reg_alpha': best['reg_alpha'],
    'random_state': 42
}



#final_model = xgb.XGBClassifier(max_depth = best['max_depth'], learning_rate = best['learning_rate'], ... , )
# Train final model with best parameters
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate
train_score = accuracy_score(y_train, final_model.predict(X_train))
test_score = accuracy_score(y_test, final_model.predict(X_test))

print(f"\nFinal Model Performance:")
print(f"Training Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")


Final Model Performance:
Training Accuracy: 0.9833
Test Accuracy: 0.9333
